In [3]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("../datasets/dataset.csv")

print("Dataset Shape:", df.shape)

# ==========================================
# CREATE SYMPTOM LIST
# ==========================================

symptom_columns = [col for col in df.columns if col.startswith("Symptom")]

symptom_lists = []

for _, row in df.iterrows():

    symptoms = []

    for col in symptom_columns:

        value = row[col]

        if pd.notna(value):

            symptoms.append(
                str(value).strip().lower()
            )

    symptom_lists.append(symptoms)

# ==========================================
# MULTILABEL BINARIZER
# ==========================================

mlb = MultiLabelBinarizer()

X = mlb.fit_transform(symptom_lists)

print("Total Symptoms:", len(mlb.classes_))

# ==========================================
# LABEL ENCODER
# ==========================================

encoder = LabelEncoder()

y = encoder.fit_transform(df["Disease"])

num_classes = len(encoder.classes_)

y = to_categorical(y)

print("Total Diseases:", num_classes)

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# BUILD MODEL
# ==========================================

model = Sequential([

    Dense(
        256,
        activation="relu",
        input_shape=(X_train.shape[1],)
    ),

    Dropout(0.3),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        num_classes,
        activation="softmax"
    )
])

# ==========================================
# COMPILE
# ==========================================

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ==========================================
# EARLY STOPPING
# ==========================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# ==========================================
# TRAIN
# ==========================================

history = model.fit(

    X_train,
    y_train,

    validation_split=0.2,

    epochs=100,

    batch_size=32,

    callbacks=[early_stop],

    verbose=1
)

# ==========================================
# EVALUATE
# ==========================================

loss, acc = model.evaluate(
    X_test,
    y_test
)

print("\nAccuracy:", round(acc * 100, 2), "%")

# ==========================================
# SAVE MODEL
# ==========================================

model.save("../models/medical_model.h5")

with open("../models/binarizer.pkl", "wb") as f:
    pickle.dump(mlb, f)

with open("../models/labels.pkl", "wb") as f:
    pickle.dump(
        encoder.classes_.tolist(),
        f
    )

print("\n✅ Model Saved Successfully")
print("medical_model.h5")
print("binarizer.pkl")
print("labels.pkl")

Dataset Shape: (4920, 18)
Total Symptoms: 131
Total Diseases: 41


C:\Users\pande\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.6356 - loss: 2.0916 - val_accuracy: 1.0000 - val_loss: 0.1093
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9914 - loss: 0.1009 - val_accuracy: 1.0000 - val_loss: 0.0058
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9987 - loss: 0.0265 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9984 - loss: 0.0165 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9984 - loss: 0.0108 - val_accuracy: 1.0000 - val_loss: 5.1427e-04
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9997 - loss: 0.0058 - val_accuracy: 1.0000 - val_loss: 8.7793e-04
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9997 - loss: 0.0050 - val_accuracy: 1.0000 - val_loss: 2.9726e-04
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9997 - loss: 0.0041 - val_accura


Accuracy: 100.0 %

✅ Model Saved Successfully
medical_model.h5
binarizer.pkl
labels.pkl


In [2]:
import pandas as pd

df = pd.read_csv("../datasets/dataset.csv")

print(df.columns)
print(df.head())

Index(['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4',
       'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9',
       'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14',
       'Symptom_15', 'Symptom_16', 'Symptom_17'],
      dtype='object')
            Disease   Symptom_1              Symptom_2              Symptom_3  \
0  Fungal infection     itching              skin_rash   nodal_skin_eruptions   
1  Fungal infection   skin_rash   nodal_skin_eruptions    dischromic _patches   
2  Fungal infection     itching   nodal_skin_eruptions    dischromic _patches   
3  Fungal infection     itching              skin_rash    dischromic _patches   
4  Fungal infection     itching              skin_rash   nodal_skin_eruptions   

              Symptom_4 Symptom_5 Symptom_6 Symptom_7 Symptom_8 Symptom_9  \
0   dischromic _patches       NaN       NaN       NaN       NaN       NaN   
1                   NaN       NaN       NaN       NaN       NaN     

In [4]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(
    y_true_classes,
    y_pred_classes
))

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        30
           2       1.00      1.00      1.00        24
           3       1.00      1.00      1.00        25
           4       1.00      1.00      1.00        24
           5       1.00      1.00      1.00        23
           6       1.00      1.00      1.00        33
           7       1.00      1.00      1.00        23
           8       1.00      1.00      1.00        21
           9       1.00      1.00      1.00        15
          10       1.00      1.00      1.00        23
          11       1.00      1.00      1.00        26
          12       1.00      1.00      1.00        21
          13       1.00      1.00      1.00        29
          14       1.00      1.00      1.00        24
          15       1.00      1.00      1.00        19
          16       1.00      1.00      1.0

In [6]:
y = pd.get_dummies(df['Disease']).values
disease_labels = pd.get_dummies(df['Disease']).columns.tolist()

In [7]:
print(df.columns)

Index(['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4',
       'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9',
       'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14',
       'Symptom_15', 'Symptom_16', 'Symptom_17'],
      dtype='object')


In [8]:
disease_labels = df["Disease"].unique().tolist()

print("Unique Diseases:", len(disease_labels))
print(disease_labels[:10])

Unique Diseases: 41
['Fungal infection', 'Allergy', 'GERD', 'Chronic cholestasis', 'Drug Reaction', 'Peptic ulcer diseae', 'AIDS', 'Diabetes ', 'Gastroenteritis', 'Bronchial Asthma']


In [10]:
import pickle

with open("../models/labels.pkl", "rb") as f:
    disease_labels = pickle.load(f)

print("Unique Diseases:", len(disease_labels))

Unique Diseases: 41


In [13]:
import os

print(os.getcwd())

C:\Users\pande\OneDrive\Desktop\Ai-Medical\train


In [14]:
model = tf.keras.models.load_model("../models/medical_model.h5")

with open("../models/binarizer.pkl", "rb") as f:
    mlb = pickle.load(f)

with open("../models/labels.pkl", "rb") as f:
    disease_labels = pickle.load(f)

In [15]:
pred = model.predict(X)

154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [16]:
model = tf.keras.models.load_model(
    "../models/medical_model.h5",
    compile=False
)

In [17]:
import numpy as np
import pickle
import tensorflow as tf

model = tf.keras.models.load_model(
    "../models/medical_model.h5",
    compile=False
)

with open("../models/binarizer.pkl", "rb") as f:
    mlb = pickle.load(f)

with open("../models/labels.pkl", "rb") as f:
    labels = pickle.load(f)

symptoms = [
    "itching",
    "skin_rash",
    "nodal_skin_eruptions"
]

X = mlb.transform([symptoms])

pred = model.predict(X)

idx = np.argmax(pred)

print("Disease:", labels[idx])
print("Confidence:", round(float(pred[0][idx]) * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
Disease: Fungal infection
Confidence: 100.0 %
